In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re

In [ ]:
# Load dataset
df = pd.read_csv("C:\Datasets\Reviews.csv")

In [ ]:
df.head()

# Exploratory Data Analysis

In [ ]:
# Drop missing values
df = df.dropna(subset=['Text'])

In [ ]:
# Remove duplicates
df = df.drop_duplicates()

In [ ]:
# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    return text
    
df['Clean_Text'] = df['Text'].apply(clean_text)


In [ ]:
# Create sentiment label
df['Sentiment'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)

In [ ]:
# Review length
df['Review_Length'] = df['Clean_Text'].apply(len)

In [ ]:
# Plot review length distribution
plt.hist(df['Review_Length'], bins=50)
plt.title("Review Length Distribution")
plt.show()

In [ ]:
bf = df['Score'].value_counts().sort_index()\
.plot(kind='bar', title='Count of Reviews by Stars', figsize=(10,5))

bf.set_xlabel('Review Stars')
plt.show()

In [ ]:
# Word Cloud
text = " ".join(df['Clean_Text'].sample(10000))
wordcloud = WordCloud(width=800, height=400).generate(text)

In [ ]:
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

# Feature Engineering

In [ ]:
#Importing TfidfVectorizer from scikit-learn, to convert a collection of text documents into a matrix of TF-IDF 
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Convert text to numerical features
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)

X = tfidf.fit_transform(df['Clean_Text'])
y = df['Sentiment']

# Train the Model

In [ ]:
#Import Utilities to Divide the dataset into 2 and Classifying discrete data such as text word counts
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [ ]:
# Train model
model = MultinomialNB(alpha=1.0)
model.fit(X_train, y_train)

In [ ]:
# Predictions
y_pred = model.predict(X_test)

# Evaluate Model

In [ ]:
# Import functions to evaluate classification model performance
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

# Interpretation:

 - The baseline model achieved accuracy of 83.78%. This indicates that the model indentifies sentiments of 83.78 out of 100 reviews
 - The model shows high precision for positive reviews, but may have lower recall for negative reviews if vocabulary is limited.
 - The analysis identifies that the model occassionaly struggles with negation, changing the meaning of these words to positive.

# Model Strength and Weakness
 - **Strength** - The model is efficient and scalable. This makes it capable to process thousands of records.
 - **Weakness** - Since the model treats every word independently, it might miss sarcastic words in the reviews.
     
# Business Meaning:

 - Helps identify unhappy customers
 - Enables targeted improvements
---

# Model Improvement
 - Improvement using n-grams (1,2)

In [ ]:
#tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=7000)
# TF-IDF with unigrams + bigrams
tfidf_bi = TfidfVectorizer(stop_words='english', max_features=7000, ngram_range=(1,2))
X_bi = tfidf_bi.fit_transform(df['Clean_Text'])

In [ ]:
# Train-test split (same random state for fairness)
X_train_bi, X_test_bi, y_train, y_test = train_test_split(
    X_bi, df['Sentiment'], test_size=0.2, random_state=42)

In [ ]:
# Train model
model_bi = MultinomialNB(alpha=1.0)
model_bi.fit(X_train_bi, y_train)

In [ ]:
# Predictions
y_pred_bi = model_bi.predict(X_test_bi)

In [ ]:
# Evaluation
from sklearn.metrics import accuracy_score, classification_report
print("\n=== IMPROVED MODEL (N-grams 1,2) ===")
print("Accuracy:", accuracy_score(y_test, y_pred_bi))
print(classification_report(y_test, y_pred_bi))

In [ ]:
# =========================
# PERFORMANCE COMPARISON
# =========================

print("\n=== PERFORMANCE COMPARISON ===")
print("Baseline Accuracy:", accuracy_score(y_test, y_pred))
print("Improved Accuracy:", accuracy_score(y_test, y_pred_bi))

---
**What was changed?**
 - While the original model looked only at single words (unigrams), the improved version looks at both single words and pairs of adjacent words (bigrams).

**Why was this change made?**

 - Single words often fail to capture the true sentiment when modifiers are used. By using bigrams, the model can "see" phrases like "not good" or "very bad" as single units of meaning.

**Impact on Result**

 - Baseline Accuracy: 83.78%
 - Improved Accuracy: 85.38%

The 1.6% increase in accuracy directly correlates to the model's new ability to correctly interpret negative feedback that uses common phrases. This ensures fewer customer complaints are misclassified as positive, allowing the business to address genuine product issues more effectively.
